# 6.0. Введение в машинное обучение

Практическая работа по методичке `module-6-0/metodichka.md`.

**ФИО:**

**Дата:**

Запускайте ноутбук из корня курса. Ядро — интерпретатор `.venv`.


## 6.0.1. Как модель получает ответ

Квартиры до 60 м² и просторнее 60 м² дают два средних. Ошибка правила — среднее модуля отклонения от своего среднего.


In [ ]:
import pandas as pd

raw = pd.read_csv("module-6-0/data/flats.csv", encoding="utf-8")
print(raw.shape)
print(raw.head(3))

left = raw.loc[raw["area"] <= 60, "price"]
right = raw.loc[raw["area"] > 60, "price"]
print("среднее до 60 м2", left.mean())
print("среднее больше 60 м2", right.mean())

pred_rule = raw["area"].apply(lambda area: left.mean() if area <= 60 else right.mean())
mae_rule = (raw["price"] - pred_rule).abs().mean()
print("ошибка правила", round(mae_rule))


## 6.0.2. Разведка таблицы


In [ ]:
print(raw.dtypes)
print("пропуски", int(raw.isna().sum().sum()))
print(raw[["area", "year", "rooms", "price"]].describe())
print("средняя цена", raw["price"].mean())
print("сумма цен", int(raw["price"].sum()))
print("сумма площадей", int(raw["area"].sum()))


Почему столбец `price_note` нельзя ставить в признаки:


## 6.0.3. Первая модель


In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

features = ["area", "year", "rooms"]
X = raw[features].copy()
y = raw["price"].copy()

model = DecisionTreeRegressor(random_state=1)
model.fit(X, y)
pred_train = model.predict(X)

print(X.shape, y.shape, pred_train.shape)
print(pred_train[:3])
print("ошибка на тех же строках", mean_absolute_error(y, pred_train))


## 6.0.4. Проверка на отложенных строках


In [ ]:
from sklearn.model_selection import train_test_split

train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)
print(train_X.shape, val_X.shape)

val_model = DecisionTreeRegressor(random_state=1)
val_model.fit(train_X, train_y)
val_pred = val_model.predict(val_X)

mae_train = mean_absolute_error(train_y, val_model.predict(train_X))
mae_val = mean_absolute_error(val_y, val_pred)
print("MAE обучение", mae_train)
print("MAE проверка", mae_val)


Какое число MAE показать заказчику:


## 6.0.5. Число листьев


In [ ]:
def mae_for_leaves(max_leaf_nodes):
    tree = DecisionTreeRegressor(max_leaf_nodes=max_leaf_nodes, random_state=1)
    tree.fit(train_X, train_y)
    return mean_absolute_error(val_y, tree.predict(val_X))

candidates = [2, 4, 6, 8]
scores = {leaves: mae_for_leaves(leaves) for leaves in candidates}
for leaves, score in scores.items():
    print(leaves, round(score, 1))

best_leaves = min(scores, key=scores.get)
print("листьев", best_leaves, "MAE", round(scores[best_leaves], 1))


Четыре MAE и выбранное число листьев. Что делает дерево из двух листьев:


## 6.0.6. Случайный лес


In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest = RandomForestRegressor(random_state=1)
forest.fit(train_X, train_y)
forest_mae = mean_absolute_error(val_y, forest.predict(val_X))
print("MAE лес", round(forest_mae, 1))
print("MAE лучшее дерево", round(scores[best_leaves], 1))
print("число деревьев", forest.n_estimators)


Какая MAE меньше — у леса или у дерева с выбранным числом листьев:


## 6.0.7. Прогноз для новых квартир


In [ ]:
from pathlib import Path

new_flats = pd.DataFrame(
    [
        {"flat_id": 101, "area": 45, "year": 1988, "rooms": 2},
        {"flat_id": 102, "area": 88, "year": 2015, "rooms": 3},
    ]
)
new_X = new_flats[features]
new_price = forest.predict(new_X)

submission = pd.DataFrame(
    {"flat_id": new_flats["flat_id"], "price": new_price.round(1)}
)
out = Path("module-6-0/data/new_prices.csv")
submission.to_csv(out, index=False, encoding="utf-8")
print(submission)
print("столбцы как у X", list(new_X.columns) == features)


## 6.0.8. Самопроверка

1. Почему MAE на строках обучения равна 0 и что из этого следует для новой квартиры.
2. Чем дерево с двумя листьями отличается от дерева с листом на каждую учебную квартиру.
3. Откуда у случайного леса одно число цены.
